<div style="background-color:#7A0019; color:white; padding:28px 32px; border-radius:12px;">
  <div style="font-size:1.55em; font-weight:700; line-height:1.3; margin-bottom:10px;">
    RegionLM: Integrating Point, Line, and Polygon Context for Urban Representation Learning
  </div>
  <div style="font-size:1.05em; color:#FFCC33; line-height:1.4;">
    A teaching module for building region embeddings from OpenStreetMap features with SpaBERT
  </div>
</div>

<br>

**Author:** Yao-Yi Chiang, Yijun Lin, Zekun Li, University of Minnesota  
**Dataset:** OpenStreetMap (OSM) features for New York City — points of interest (POIs), buildings, and land use  
**Task:** Learn urban region representations by encoding nearby spatial context into SpaBERT-style contextual embeddings  
**Model:** SpaBERT (spatial BERT) with region-level aggregation and clustering  
**Platform:** Local Python environment (Python 3.10+) or Jupyter — see Prerequisites below for setup

---

RegionLM is a geospatial representation learning pipeline built around SpaBERT-style contextual embeddings for OpenStreetMap (OSM) features. It extracts features inside target regions, converts nearby spatial context into pseudo-sentences, trains or applies a spatial BERT model, aggregates feature embeddings into region embeddings, and clusters the resulting regions.

The repository is organized as a script-driven research workflow. This notebook walks through the intended end-to-end sequence, while the numbered Python scripts provide CLI entrypoints for each stage.

### Learning objectives

By the end of this notebook, you will be able to:

- Set up the RegionLM repository, Python environment, and OSM datasets needed for the pipeline
- Clip OSM point, polygon, and land-use features to a study region and export them for downstream use
- Rasterize polygon features into H3-based area-of-interest (AOI) points and merge AOI layers
- Generate SpaBERT pseudo-sentence inputs that combine POIs with nearby spatial context
- Train or apply SpaBERT to produce feature embeddings, then aggregate them into region embeddings
- Reduce embedding dimensions and cluster regions to explore urban structure


<hr style="border:none; border-top:2.5px solid #7A0019; margin:28px 0 8px 0;">


## Prerequisites

Before running the pipeline, set up the repository, environment, and datasets.

### 1. GitHub Repository

The source code is available at:

https://github.com/knowledge-computing/ucgis-regionlm

Clone the repository to your local machine:

```bash
git clone https://github.com/knowledge-computing/ucgis-regionlm.git
cd ucgis-regionlm
```

### 2. Environment

This project targets Python 3.10+ and depends on PyTorch, Hugging Face Transformers, GeoPandas, Shapely, H3, and related geospatial tooling.

Create an environment and install dependencies:

```bash
conda create --name py310 -y python=3.10
conda activate py310
pip install -r requirements.txt
pip install jupyter
```

Notes:
- GeoPandas may require system libraries such as GDAL/GEOS/PROJ depending on your platform.
- Training and embedding generation will use CUDA if PyTorch detects a GPU.

### 3. Datasets and Pretrained Weights

The current workflow expects external datasets and model weights:

- [Datasets and Model weights](https://drive.google.com/drive/folders/1eCsvW92ZWvJDkDsUPSaNBb9tpd0ZNmsW?usp=sharing)

What you need is the raw shapefiles from OpenStreetMap:

- region boundaries
- OSM POIs
- OSM buildings
- OSM land use

The intermediate output of each step are also available.

The notebook examples assume a `data/` directory with paths such as:

```text
data/
  nyc_regions/region.shp
  gis_osm_pois_free_1/gis_osm_pois_free_1.shp
  gis_osm_buildings_a_free_1/gis_osm_buildings_a_free_1.shp
  gis_osm_landuse_a_free_1/gis_osm_landuse_a_free_1.shp
```


<hr style="border:none; border-top:2.5px solid #7A0019; margin:28px 0 8px 0;">


## Repository Layout

- [`0_regionlm_tutorial.ipynb`](https://github.com/knowledge-computing/ucgis-regionlm/blob/main/0_regionlm_tutorial.ipynb): notebook walkthrough for the entire pipeline, including data preprosessing and modeling
- [`1_1_extract_osm_features_in_region.py`](https://github.com/knowledge-computing/ucgis-regionlm/blob/main/1_1_extract_osm_features_in_region.py): clip OSM shapefiles to the study region and export CSV
- [`1_2_rasterization.py`](https://github.com/knowledge-computing/ucgis-regionlm/blob/main/1_2_rasterization.py): convert polygon features to H3/geohash AOI points
- [`1_3_generate_spabert_json.py`](https://github.com/knowledge-computing/ucgis-regionlm/blob/main/1_3_generate_spabert_json.py): create pseudo-sentence JSONL for SpaBERT
- [`2_1_train_predict_spabert.py`](https://github.com/knowledge-computing/ucgis-regionlm/blob/main/2_1_train_predict_spabert.py): train SpaBERT or generate feature embeddings
- [`2_2_region_embedding.py`](https://github.com/knowledge-computing/ucgis-regionlm/blob/main/2_2_region_embedding.py): aggregate feature embeddings into region embeddings
- [`2_3_dimension_reduction_clustering.py`](https://github.com/knowledge-computing/ucgis-regionlm/blob/main/2_3_dimension_reduction_clustering.py): dimension reduction and KMeans clustering
- [`spabert/`](https://github.com/knowledge-computing/ucgis-regionlm/tree/main/spabert): SpaBERT model and dataset utilities
- [`utils/const.py`](https://github.com/knowledge-computing/ucgis-regionlm/blob/main/utils/const.py): shared field names and default region settings


<hr style="border:none; border-top:2.5px solid #7A0019; margin:28px 0 8px 0;">


## Pipeline Overview

The following steps form the end-to-end RegionLM workflow. Work through them in order; each step lists learning objectives and the commands or code to run.

1. Download OSM data for the study area (New York).
2. Extract OSM features that intersect a target region.
3. Rasterize polygons such as buildings or land use into H3 area-of-interest (AOI) points, then merge AOI files.
4. Build SpaBERT pseudo-sentence JSON from POIs plus optional AOI context.
5. Train SpaBERT or load existing model weights to generate POI embeddings.
6. Aggregate POI embeddings into region-level embeddings.
7. Optionally reduce dimensions and cluster the resulting regions.


<hr style="border:none; border-top:1px dashed #C8C8C8; margin:18px 0;">


### Step 1: Download OSM data in New York

**Goal**
- Identify which OSM feature types RegionLM uses (buildings, land use, and POIs).
- Obtain the required New York datasets and arrange them under the expected `./data` folder structure.

In this study, we focus on **Buildings**, **Landuse**, and **POI**.

You can download OSM extracts from [Geofabrik](https://download.geofabrik.de/).

We have also provided the required data: [Drive folder](https://drive.google.com/drive/folders/13l2ZczVrcWhVbTY1e_xvNiAUvWG1ovh-?usp=drive_link).

Follow the folder structure under `./data` described in Prerequisites.


<hr style="border:none; border-top:1px dashed #C8C8C8; margin:18px 0;">


### Step 2: Extract features within the study region (NYC)

**Goal**
- Clip OSM shapefiles to the NYC study region (with nyc_regions/region.shp).
- Export region-filtered buildings, land use, and POIs as both shapefile and CSV for later steps.

```bash
python 1_1_extract_osm_features_in_region.py \
    --region_shp "data/nyc_regions/region.shp" \
    --osm_feature_shp "data/gis_osm_pois_free_1/gis_osm_pois_free_1.shp" \
    --output_shp "data/nyc_osm_pois/pois.shp" \
    --output_csv "data/nyc_osm_pois/pois.csv"

python 1_1_extract_osm_features_in_region.py \
    --region_shp "data/nyc_regions/region.shp" \
    --osm_feature_shp "data/gis_osm_buildings_a_free_1/gis_osm_buildings_a_free_1.shp" \
    --output_shp "data/nyc_osm_buildings/buildings.shp" \
    --output_csv "data/nyc_osm_buildings/buildings.csv"

python 1_1_extract_osm_features_in_region.py \
    --region_shp "data/nyc_regions/region.shp" \
    --osm_feature_shp "data/gis_osm_landuse_a_free_1/gis_osm_landuse_a_free_1.shp" \
    --output_shp "data/nyc_osm_landuse/landuse.shp" \
    --output_csv "data/nyc_osm_landuse/landuse.csv"
```


<hr style="border:none; border-top:1px dashed #C8C8C8; margin:18px 0;">


### Step 3: Rasterize features with H3

**Goal**
- Convert polygon features (buildings and land use) into H3-based AOI point representations.
- Merge the rasterized AOI CSVs into a single `aoi.csv` file for SpaBERT input generation.

```bash
python 1_2_rasterization.py \
    --input_csv_path data/nyc_osm_buildings/buildings.csv \
    --output_csv_path data/nyc_buildings_h3.csv

python 1_2_rasterization.py \
    --input_csv_path data/nyc_osm_landuse/landuse.csv \
    --output_csv_path data/nyc_landuse_h3.csv
```

#### Merge AOI files

Run the next cell to concatenate the H3 building and land-use outputs into `data/aoi.csv`.


In [ ]:
import os
import pandas as pd
from typing import List, Optional

def concat_csv_files(
    file_paths: List[str],
    output_file: str,
    columns: Optional[List[str]] = None,
    ignore_index: bool = True,
    add_source: bool = False,
    encoding: Optional[str] = None
) -> pd.DataFrame:
    """
    Concatenate multiple CSV files, optionally keeping only selected columns,
    and save the result to an output CSV file.

    Args:
        file_paths: List of input CSV file paths.
        output_file: Path to the output CSV file.
        columns: List of columns to keep. If None, keep all columns.
        ignore_index: Whether to reset the index in the concatenated result.
        add_source: Whether to add a column with the source file name.
        encoding: Optional file encoding for reading/writing CSVs.

    Returns:
        The concatenated DataFrame.
    """
    dfs = []

    for fp in file_paths:
        if not os.path.exists(fp):
            print(f"Warning: {fp} not found, skipping.")
            continue

        df = pd.read_csv(fp, encoding=encoding)

        if columns is not None:
            missing_cols = [col for col in columns if col not in df.columns]
            if missing_cols:
                print(f"Warning: {fp} is missing columns {missing_cols}, skipping.")
                continue
            df = df[columns]

        if add_source:
            df["source_file"] = os.path.basename(fp)

        dfs.append(df)

    if not dfs:
        raise ValueError("No valid CSV files found.")

    result = pd.concat(dfs, ignore_index=ignore_index)

    output_dir = os.path.dirname(output_file)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    result.to_csv(output_file, index=False, encoding=encoding)

    return result


files = ["data/nyc_buildings_h3.csv", "data/nyc_landuse_h3.csv"]

data = concat_csv_files(
    file_paths=files,
    output_file="data/aoi.csv",
    columns=["osm_id", "agg_category", "geometry", "h3_list", "poi_aoi"],
    add_source=False
)
data


,osm_id,agg_category,geometry,h3_list,poi_aoi
0,31974497,residential,POINT (-73.9673843946306 40.67906738559073),8b2a100da205fff,aoi
1,34633854,office:empire state building,POINT (-73.9850578581105 40.74843122871664),8b2a100d2d49fff,aoi
2,34633854,office:empire state building,POINT (-73.98598888616235 40.74878374676755),8b2a100d2d4afff,aoi
3,34633854,office:empire state building,POINT (-73.98534848824615 40.748045690473624),8b2a100d2896fff,aoi
4,34633854,office:empire state building,POINT (-73.985668684648 40.74841471816488),8b2a100d2d4bfff,aoi
...,...,...,...,...,...
72575,1137610816,retail,POINT (-73.76218265152225 40.74782058110505),8b2a100504d1fff,aoi
72576,1137610818,forest,POINT (-73.75784786501667 40.75510402345344),8b2a1005626bfff,aoi
72577,1137610818,forest,POINT (-73.75752947378393 40.75473437010585),8b2a10050536fff,aoi
72578,1137610818,forest,POINT (-73.75816626134745 40.75547367773417),8b2a1005626afff,aoi


<hr style="border:none; border-top:1px dashed #C8C8C8; margin:18px 0;">


### Step 4: Generate SpaBERT input

**Goal**
- Combine clipped POIs with the merged AOI context.
- Produce pseudo-sentence JSON that SpaBERT can consume for training or prediction.

```bash
python 1_3_generate_spabert_json.py \
    --csv_file_path data/nyc_osm_pois/pois.csv \
    --processed_aoi_csv_file_path data/aoi.csv
```


<hr style="border:none; border-top:1px dashed #C8C8C8; margin:18px 0;">


### Step 5: Train SpaBERT and generate embeddings

**Goal**
- Train a SpaBERT model on the pseudo-sentence JSON (or skip training if weights already exist).
- Generate POI-level SpaBERT embeddings for the study region.

Train:

```bash
python 2_1_train_predict_spabert.py \
    --json_file_path='data/nyc_osm_pois/pois/pois_pseudo_sentence_v2_nn100_sdm100.json' \
    --model_save_dir='model_weights/' \
    --mode='train'
```

Predict / generate embeddings:

```bash
python 2_1_train_predict_spabert.py \
    --json_file_path='data/nyc_osm_pois/pois/pois_pseudo_sentence_v2_nn100_sdm100.json' \
    --model_save_dir='model_weights/' \
    --csv_file_path='data/nyc_osm_pois/pois/pois_pseudo_sentence_v2_nn100_sdm100_spabert_embedding.csv' \
    --mode='predict'
```


<hr style="border:none; border-top:1px dashed #C8C8C8; margin:18px 0;">


### Step 6: Aggregate SpaBERT embeddings into region embeddings

**Goal**
- Group POI-level SpaBERT embeddings by region.
- Produce a region embedding CSV for downstream analysis.

```bash
python 2_2_region_embedding.py \
    --in_csv_file_path='data/nyc_osm_pois/pois/pois_pseudo_sentence_v2_nn100_sdm100_spabert_embedding.csv' \
    --out_csv_file_path='data/nyc_osm_pois/pois/pois_pseudo_sentence_v2_nn100_sdm100_region_embedding.csv'
```


<hr style="border:none; border-top:1px dashed #C8C8C8; margin:18px 0;">


### Step 7: Dimension reduction and clustering

**Goal**
- Reduce the dimensionality of region embeddings for visualization or analysis.
- Cluster regions (e.g., with KMeans) and export the cluster assignments.

```bash
python 2_3_dimension_reduction_clustering.py \
    --input_region_embedding_csv='data/nyc_osm_pois/pois/pois_pseudo_sentence_v2_nn100_sdm100_region_embedding.csv' \
    --output_dimension_reduce_csv='data/nyc_osm_pois/pois/pois_pseudo_sentence_v2_nn100_sdm100_dimreduce.csv' \
    --output_cluster_csv='data/nyc_osm_pois/pois/pois_pseudo_sentence_v2_nn100_sdm100_cluster.csv'
```


### Step 8: Cluster Visualization

**Goal**
- Visualize the spatial distribution of clustered regions on a map.
- Color each region by its assigned cluster to facilitate interpretation of regional patterns.

```bash
python 3_0_visualize_cluster.py \
    --input_cluster_csv data/nyc_osm_pois/pois/pois_pseudo_sentence_v2_nn100_sdm100_cluster.csv \
    --output_png data/nyc_osm_pois/h3_clusters.png
```